In [1]:
pygame_or_nah = False
verbose_or_nah = False

In [2]:
steps = 150
new_world = 20

gamma_values = [0.95]
lr_values = [5e-7]
gae_lambda_values = [0.95]
policy_clip_values = [0.2]
batch_size_values = [64]
n_epochs_values = [5]
#LSTM_hyperparam_Sweep_1__g0.95_lr5e-07_lmb0.95_clip0.2_bs64_ep5

reward_type = 1 #[1, 0, -1]

model_name = 'name_test_2_'#'Random_Run' 
load = True ### Load existing models or start fresh.   ## Remember to give a new model name if you're switching.

In [3]:
#!pip install pygame
#!pip install opencv-python

In [4]:
import os
import sys
#sys.path.append('/content/drive/MyDrive/Noli')
from Environment import Environment
from Council import TheCouncil

import random
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

if pygame_or_nah:
    import pygame

#os.environ["SDL_VIDEODRIVER"] = "dummy"                  #  for pygamee window

In [5]:
def convert_index(index):
    value = index % 6
    x = (index // 6) % 10
    y = (index // 6) // 10
    return [value, x, y]

In [6]:
import cv2
#from google.colab.patches import cv2_imshow
#from google.colab import output
import time
import torch

width = 1000
height = 1000

FPS = 20


#pygame.init()
#DISP = pygame.display.set_mode((width, height))

if pygame_or_nah:
    clock = pygame.time.Clock( )

Sample = np.zeros((width, height), dtype = np.uint32)
#Output = np.zeros_like(Sample)

# byte packing function
def Pack4bytes(c1,c2,c3,c4) :
    result = c1 << 24 | c2 << 16 | c3 << 8 | c4
    return result

#Initialise color channels
A = np.zeros_like(Sample)
R = np.zeros_like(Sample)
G = np.zeros_like(Sample)
B = np.zeros_like(Sample)

def pygameStep(screen):
    if pygame_or_nah:
          pygame.display.flip()
          screen.fill((0,0,0))
          #convert image so it can be displayed in OpenCV
          view = pygame.surfarray.array3d(screen)
            
          #  convert from (width, height, channel) to (height, width, channel)
          view = view.transpose([1, 0, 2])

          #  convert from rgb to bgr
          img_bgr = cv2.cvtColor(view, cv2.COLOR_RGB2BGR)

          #Display image, clear cell every 0.5 seconds
          #cv2.imshow(view, img_bgr)
          time.sleep(0.5)
          #utput.clear()
          #

def Render(reward, land_tensors, resource_tensors, reward_counter):


    if type(land_tensors) == type(torch.tensor(1)):

      buildings = np.array(land_tensors[3])
      land_vals = np.array(land_tensors)  

      A = np.repeat(np.repeat(np.zeros([10,10]), 100, axis=0), 100, axis=1).astype(int)

      upscaled_G = np.repeat(np.repeat(land_vals[0], 100, axis=0), 100, axis=1) ## Food
      upscaled_B = np.repeat(np.repeat(land_vals[1], 100, axis=0), 100, axis=1)  ## Mine
      upscaled_R = np.repeat(np.repeat(land_vals[2], 100, axis=0), 100, axis=1) ## Wood

      # Scale the values to the range [0, 255]
      R = ((upscaled_R * 255) / 10).astype(int)
      G = ((upscaled_G * 255) / 10).astype(int)
      B = ((upscaled_B * 255) / 10).astype(int)


      Output = Pack4bytes(A,R,G,B)


      # blit Output to display, this one is for output same size as display
      pygame.surfarray.blit_array(game_surface, Output)



      coordinates = np.array(land_tensors[3].nonzero())

      farm_img = pygame.transform.scale(pygame.image.load("Assets/Farm.png").convert_alpha(), (100, 100))
      mine_img = pygame.transform.scale(pygame.image.load("Assets/Mine.png").convert_alpha(), (100, 100))
      wood_img = pygame.transform.scale(pygame.image.load("Assets/Lumber.png").convert_alpha(), (100, 100))
      house_img = pygame.transform.scale(pygame.image.load("Assets/House.png").convert_alpha(), (100, 100))
      latrine_img = pygame.transform.scale(pygame.image.load("Assets/Latrine.png").convert_alpha(), (100, 100))


      print(np.array(pygame.surfarray.array3d(farm_img)).shape)


      img = None


      for coord in coordinates:

          x = coord[0]
          y = coord[1]

          if buildings[x][y]== 0:
              pass

          if buildings[x][y]== 1: #here
            img = house_img

          ## Is it a farm?
          if buildings[x][y]== 2:
            img = farm_img


          ## Is it a farm?
          if buildings[x][y]== 3:
            img = mine_img

          ## Is it a farm?
          if buildings[x][y]== 4:
            img = wood_img


          if buildings[x][y] == 5:
              img = latrine_img


          if img:
            game_surface.blit(img, (coord[0] * 100, coord[1] * 100))

    if type(resource_tensors) == type(torch.tensor(1)):

        food_num = resource_tensors[0][0]
        ore_num = resource_tensors[0][1]
        wood_num = resource_tensors[0][2]
        people_num = resource_tensors[0][3]
        poop_num = resource_tensors[0][4]

        reward_text = 0
        if reward:
            reward_text = float(reward)
            
        text = font.render(f'People: {int(people_num)}, Food: {int(food_num)}, Poop: {int(poop_num)}', True, (255, 255, 255))  # True = anti-aliasing, color = white
        text_2 = font.render(f'Wood: {int(wood_num)}, Ore: {int(ore_num)}, Reward: {reward_text:.4f}', True, (255, 255, 255)) 
        text_3 = font.render(f'Reward_Annual_Total: {sum(reward_counter):.4f}', True, (255, 255, 255)) 
        text_rect = text.get_rect(center=(200, 10))  # Center of the screen
        text_rect_2 = text_2.get_rect(center=(200, 50))
        text_rect_3 = text_3.get_rect(center=(200, 80))

        if text_rect:
            info_surface.blit(text, text_rect) 
        if text_rect_2:
            info_surface.blit(text_2, text_rect_2)
        if text_rect_3:
            info_surface.blit(text_3, text_rect_3)

    scaled_surface = pygame.transform.scale(game_surface, (window_width, window_height-(100*scale)))

    DISP.blit(info_surface, (0,0))
    DISP.blit(scaled_surface, (0,100))
    

    pygameStep(screen=DISP)


In [7]:
from IPython.display import clear_output
# fun(episodes, epochs, N,reward_type)
N = 5

scale = 0.5

width = 1000
height = 1100
window_width = width * scale
window_height = height * scale

if pygame_or_nah:
    pygame.init()
    DISP = pygame.display.set_mode((window_width, window_height))
    game_surface = pygame.Surface((width, height-100))
    info_surface = pygame.Surface((width, 100))
    
    font = pygame.font.Font(None, 36)  # None = default font, 36 = size



def hyper_run(council, env, new_world=256, steps=100):
    
    land_tensors = None
    
    for i in range(new_world):
    
    
      print(f'\n *** *** *** *** \n The New Horizon Dawns for the {i}th time \n *** *** *** *** \n ')
    
      land_tensors, resource_tensors = env.reset()
    
      #council_decision = [1,0,5]
    
      reward = None
      reward_counter = []
    
      for i in range(steps):
    
        if pygame_or_nah:
            clock.tick(FPS)
            game_surface.fill((0,0,0))
            info_surface.fill((0,0,0))
            Render(reward, land_tensors, resource_tensors, reward_counter)
    
        if env.end == True:
          break
            
        if reward_type != 0:
          council_decision, prob, critic_val = the_council.choose_action(land_tensors, resource_tensors)
        else:
          council_decision = random.randint(0,500)
    
        council_plans = convert_index(council_decision)
        land_tensors_, resource_tensors_ = env.step(council_plans) #reward
    
        reward = reward_type * env.reward  ## hash out reward_type to record negative reward.
        reward_counter.append(reward)
          
        if reward_type != 0:
          #score += reward # plot reward as we go
          the_council.remember([land_tensors, resource_tensors], council_decision, prob, critic_val, reward, env.end)
    
          if i+1 % N == 0:
                    the_council.learn()
                    #learn_iters += 1
    
        land_tensors, resource_tensors = land_tensors_, resource_tensors_
    
    
    
    
      ### Save
    
      the_council.save_models()
    
      the_council.reset_memory()  ## For LSTM
    
    
      def graph_metrics(title, labels, *metrics):
    
            for idx, metric in enumerate(metrics):
              plt.plot(metric, label = labels[idx])
            plt.title(title)
            plt.legend()
            plt.show()
    
    
        
    
    
      def save_metrics(model_id, filename, labels, *metrics):
    
            folder = os.path.join('Run_Data', model_id)
            os.makedirs(folder, exist_ok=True)
        
            base_path = os.path.join(folder, f'{filename}.csv')
            run_id = 1
        
            if os.path.exists(base_path):
                existing_df = pd.read_csv(base_path)
                run_id = existing_df['run_id'].max() + 1 if 'run_id' in existing_df.columns else 1
            else:
                existing_df = pd.DataFrame()
        
            new_df = pd.DataFrame({label: metric for label, metric in zip(labels, metrics)})
            new_df['run_id'] = run_id
        
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            combined_df.to_csv(base_path, index=False)
        
    
      if verbose_or_nah:
          graph_metrics('Buildings', [ 'houses', 'farms', 'mines', 'lumber mills', 'latrines'], env.houses_list, env.farms_list, env.mines_list, env.lumber_mills_list, env.latrines_list)
        
          graph_metrics('Resources', ['food', 'ore', 'wood','poop'], env.food_list, env.ore_list, env.wood_list, env.poop_list)
        
          graph_metrics('Food produced vs eaten', ['Food Produced', 'Food Eaten', 'Current Food'], env.food_produced_list, env.food_eaten_list, env.food_list)
        
          graph_metrics('People', ['population', 'babies'], env.people_list, env.baby_list)
        
        
          graph_metrics('Reward', ['Time'], np.cumsum(reward_counter))

      else:
          print('Reward:')
          print(np.cumsum(reward_counter)[-1])
    
      save_metrics(
          model_id,
          'buildings',
          ['houses', 'farms', 'mines', 'lumber mills', 'latrines'],
          env.houses_list,
          env.farms_list,
          env.mines_list,
          env.lumber_mills_list,
          env.latrines_list
      )
    
      save_metrics(
          model_id,
          'resources',
          ['food', 'ore', 'wood', 'poop'],
          env.food_list,
          env.ore_list,
          env.wood_list,
          env.poop_list
      )
    
      save_metrics(
          model_id,
          'food',
          ['Food Produced', 'Food Eaten', 'Current Food'],
          env.food_produced_list,
          env.food_eaten_list,
          env.food_list
      )
    
      save_metrics(
          model_id,
          'people',
          ['population', 'babies'],
          env.people_list,
          env.baby_list
      )
    
      save_metrics(
          model_id,
          'reward',
          ['Time'],
          np.cumsum(reward_counter)
      )


In [8]:
import itertools

# Define hyperparameter grid


#gamma_values = [0.95, 0.975, 0.99]
#lr_values = [5e-6, 5e-7, 5e-8, 5e-9]
#gae_lambda_values = [0.90, 0.95]
#policy_clip_values = [0.1, 0.2]
#batch_size_values = [64]
#n_epochs_values = [5]
# 3 x 4 x 2  x 2 = 44 
sweep = list(itertools.product(
    gamma_values,
    lr_values,
    gae_lambda_values,
    policy_clip_values,
    batch_size_values,
    n_epochs_values
))

for idx, (gamma, lr, gae_lambda, policy_clip, batch_size, n_epochs) in enumerate(sweep):
    print(f"\n=== Running combo {idx+1}/{len(sweep)} ===")
    print(f"gamma={gamma}, lr={lr}, gae_lambda={gae_lambda}, policy_clip={policy_clip}, batch_size={batch_size}, n_epochs={n_epochs}\n")

    model_id = f"{model_name}_g{gamma}_lr{lr}_lmb{gae_lambda}_clip{policy_clip}_bs{batch_size}_ep{n_epochs}"

    the_council = TheCouncil(
        model_name=model_id,
        gamma=gamma,
        lr=lr,
        gae_lambda=gae_lambda,
        policy_clip=policy_clip,
        batch_size=batch_size,
        n_epochs=n_epochs
    )

    if the_council.check_for_checkpoints():
        the_council.load_models(load)

    env = Environment()

    env.print = verbose_or_nah
    
    model_id = f"{model_name}_g{gamma}_lr{lr}_lmb{gae_lambda}_clip{policy_clip}_bs{batch_size}_ep{n_epochs}"

    hyper_run(the_council, env, steps=steps, new_world=new_world) 


=== Running combo 1/1 ===
gamma=0.95, lr=5e-07, gae_lambda=0.95, policy_clip=0.2, batch_size=64, n_epochs=5


 *** *** *** *** 
 The New Horizon Dawns for the 0th time 
 *** *** *** *** 
 
... saving models ...
Reward:
37.36022593399881

 *** *** *** *** 
 The New Horizon Dawns for the 1th time 
 *** *** *** *** 
 
... saving models ...
Reward:
131.61164095441467

 *** *** *** *** 
 The New Horizon Dawns for the 2th time 
 *** *** *** *** 
 
... saving models ...
Reward:
21.63888068760313

 *** *** *** *** 
 The New Horizon Dawns for the 3th time 
 *** *** *** *** 
 
... saving models ...
Reward:
110.75289806969893

 *** *** *** *** 
 The New Horizon Dawns for the 4th time 
 *** *** *** *** 
 
... saving models ...
Reward:
85.04719238486864

 *** *** *** *** 
 The New Horizon Dawns for the 5th time 
 *** *** *** *** 
 
... saving models ...
Reward:
26.417885085724155

 *** *** *** *** 
 The New Horizon Dawns for the 6th time 
 *** *** *** *** 
 
... saving models ...
Reward:
149.07145